<a href="https://colab.research.google.com/github/vectara/example-notebooks/blob/main/notebooks/api-examples/12-agent-skills.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Vectara Agent Skills: Progressive-Disclosure Instructions

This notebook shows how to give a Vectara agent **skills** — specialist instruction sets that live at the agent level but only enter the conversation context when invoked.

A skill has two parts:

- **`description`** (≤ 500 chars) — a short hint shown in the system prompt every turn. The model reads this to decide whether the skill is relevant.
- **`content`** (≤ 50,000 chars) — the full specialist instructions, lazy-loaded into context only when the skill is invoked.

That's the pattern: keep the system prompt small, surface the heavy guidance only when the situation calls for it.

You'll learn how to:
1. Configure an agent with a `skills` map
2. Watch the agent autonomously invoke a skill via the auto-exposed `invoke_skill` tool
3. Trigger a skill from the client (input message of type `"skill"`) to preload context
4. Define multiple skills and let the model pick the right one per question
5. Restrict skills per step using `allowed_skills`
6. Decide between a skill, a tool, a sub-agent, and just-stuffing-the-prompt

We'll build a **Support Copilot** — where teams already use the platform to ground answers in support docs and policy. Skills slot in as the *workflow* layer on top: which playbook should the copilot adopt for *this particular* customer message?

## Skills vs. tools vs. just stuffing the prompt

| Construct | What it gives you | When to use |
|---|---|---|
| **Skill** | Specialist *instructions* loaded only when needed | You have multiple distinct mindsets/runbooks/voices the agent might need; stuffing them all into the system prompt would balloon every turn's token cost. |
| **Tool** (e.g. `web_get`, `lambda`, `corpora_search`) | A *capability* — the agent can take an action | The agent needs to **do** something (fetch data, search a corpus, call an API), not just adopt a different mindset. |
| **System-prompt baseline** | Always-on instructions | The guidance applies to every turn and is short enough that loading-on-demand isn't worth the indirection. |
| **Sub-agent** | A wholly separate agent invoked as a tool | The work needs its own model, its own tool set, or its own multi-step flow. Heavier than a skill — see notebook 5. |

The clearest tell that you want a *skill*, not a tool or a longer prompt: you're about to write *"… and ALSO, when the user reports an outage, do these 30 things …"* in your system prompt. Pull each "and ALSO" branch out into a skill instead.

### How invocation works

When an agent has skills configured, Vectara automatically exposes an `invoke_skill` tool to the model. There are two ways a skill ends up loaded:

1. **Model-driven.** The LLM emits a `tool_input` for `invoke_skill` with the skill's `skill_name`. Vectara responds by injecting the skill's `content` as a user message and emitting a `skill_load` event. (Demonstrated in Step 2 below.)
2. **Client-driven.** You POST an event with `messages: [{"type": "skill", "skill_name": "…"}]`. No model round-trip needed for the decision — Vectara loads the content immediately. Useful when your UI knows up front which mode to enter (e.g. a routing rule that always loads the outage runbook for severity-1 alerts). (Demonstrated in Step 3 below.)

Either way, the loaded content stays in session history and is available for follow-up turns (until compaction, if you have it on).

## Getting Started

All you need for this notebook is a `VECTARA_API_KEY`. The notebook is self-contained — it does not depend on the corpora, agents, or tools created in earlier notebooks.

## Setup

In [1]:
import os
import json
from datetime import datetime

import requests

api_key = os.environ['VECTARA_API_KEY']

BASE_URL = "https://api.vectara.io/v2"

headers = {
    "x-api-key": api_key,
    "Content-Type": "application/json"
}

In [2]:
# Load the shared helpers (delete_and_create_agent).
# vectara_utils.py sits next to this notebook in the repo; Colab fetches it on
# first run so the same notebook works locally and in a fresh Colab runtime.
try:
    from vectara_utils import delete_and_create_agent
except ImportError:
    import urllib.request
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/vectara/example-notebooks/main/notebooks/api-examples/vectara_utils.py",
        "vectara_utils.py",
    )
    from vectara_utils import delete_and_create_agent

import vectara_utils
vectara_utils.configure(BASE_URL, headers)

## Step 1: Define an Agent with One Skill

We'll build a **Support Copilot** that, by default, replies to customer-facing messages briefly and politely. It also has one skill — `customer_escalation` — whose content is a longer structured runbook for outages and severity calls. The agent only loads that runbook when an incoming message actually looks like an escalation.

The skill itself is just `{description, content}`. The description (≤ 500 chars) is what the model reads on every turn to decide whether to invoke; the content (≤ 50,000 chars) is the heavy part that gets loaded on demand.

In [3]:
SKILL_AGENT_NAME = "Support Copilot"

# Default instructions: terse, polite support assistant. Notice we do NOT bake
# the escalation runbook in here — that lives in the skill's `content` and only
# enters context when the agent invokes the skill.
support_copilot_instructions = """You are Support Copilot, an internal assistant that helps a customer-support agent draft replies and decide what to do next.
For most messages, reply with a short, polite acknowledgement and one suggested next step (1–3 sentences total).

If the inbound message looks like an outage, an angry customer, or anything time-sensitive, you have a `customer_escalation` skill — invoke it to load the severity rubric and runbook into context, then proceed using its guidance."""

# A SKILL is just a {description, content} pair.
# - description (≤500 chars): shown in the system prompt every turn
# - content     (≤50k chars):  loaded only when the skill is invoked
customer_escalation_skill = {
    "description": (
        "Use when an inbound customer message reports an outage, a regression, "
        "data loss, or visible anger about an urgent issue. Loads the severity "
        "rubric, routing rules, and outbound message templates."
    ),
    "content": """When the inbound message looks like an escalation, follow this runbook exactly. Produce a single Markdown response with these sections, in order:

## 1. Severity classification
Pick exactly ONE level using this rubric:
- **SEV-1** — production is down for many customers OR data loss is confirmed OR security/PII exposure is suspected.
- **SEV-2** — a core feature is broken for one customer (or many customers degraded), no data loss, workaround exists.
- **SEV-3** — a non-core feature is broken or behaving unexpectedly; the customer can keep working.
- **SEV-4** — usage question, single-user friction, or feature confusion. Not actually an incident.

State the chosen level and the one or two phrases from the customer's message that anchored your call.

## 2. Required information to collect
Before promising anything, list the facts you still need from the customer. At minimum:
- Account / tenant / org id
- Time the issue started (their timezone)
- Region or environment if known
- Reproduction steps OR scope (one user vs. all users)
- Any error messages, screenshots, or request IDs they can share

## 3. Routing
Map the severity to who should be paged, using these defaults:
- SEV-1 → page on-call SRE immediately + notify the account's CSM + post in #incidents.
- SEV-2 → file a Jira ticket on the owning team and Slack the team's channel; CSM informed via email.
- SEV-3 → file a Jira ticket; no paging.
- SEV-4 → reply directly; no ticket unless the question recurs.

State the route and the channel/handle to use. Do not invent specific names if they aren't in context — write `<on-call SRE>` or `<owning team>` and let the human fill them in.

## 4. Customer-facing reply (draft)
Draft a reply the support agent can send back to the customer right now. Constraints:
- Acknowledge the impact in the customer's own words.
- State what you're doing next (investigating / paging / filing).
- Commit ONLY to the next contact time, not to a fix time. Use a concrete time window.
- Do not promise root cause, refunds, or SLAs unless you have explicit grounds.
- 4 sentences maximum.

## 5. Internal note (one paragraph)
A handoff note for whoever picks this up: severity, scope, the missing facts from §2, and any signal of churn risk you noticed. Plain text, no greeting."""
}

skill_agent_config = {
    "name": SKILL_AGENT_NAME,
    "description": "Customer-support copilot with a customer_escalation skill loaded on demand.",
    "model": {"name": "gpt-4o"},
    "first_step_name": "main",
    "steps": {
        "main": {
            "instructions": [
                {"type": "inline", "name": "support_copilot_instructions",
                 "template": support_copilot_instructions}
            ],
            "output_parser": {"type": "default"},
        }
    },
    "skills": {
        "customer_escalation": customer_escalation_skill,
    },
}

skill_agent_key = delete_and_create_agent(skill_agent_config, SKILL_AGENT_NAME)

print(f"\nSkill description (always in system prompt): {len(customer_escalation_skill['description'])} chars")
print(f"Skill content     (loaded only on invoke):   {len(customer_escalation_skill['content'])} chars")
print(f"Ratio: ~{len(customer_escalation_skill['content']) // max(len(customer_escalation_skill['description']), 1)}x bigger — that's the token-economy win.")

Created agent 'Support Copilot' (key: agt_support_copilot_efae)

Skill description (always in system prompt): 194 chars
Skill content     (loaded only on invoke):   2282 chars
Ratio: ~11x bigger — that's the token-economy win.


Notice the size ratio. Even with this small demo, content is roughly an order of magnitude bigger than description — and your real runbooks will skew further (severity matrices, on-call rotations, customer-comms templates per tier). The system prompt only carries the cheap part on every turn; the heavy runbook enters context only when the agent actually decides it's needed.

## Step 2: Autonomous Invocation

We'll send a message that obviously needs the escalation runbook, and watch the agent decide to invoke it. The interesting events to watch in the trace:

- **`tool_input`** for `invoke_skill` — the agent chose to load a skill, and which one.
- **`skill_load`** — Vectara responded by injecting the skill's `content` into context as a user message.
- **`agent_output`** — the final reply the support agent sees.

The helper below pulls those out so you can see the lifecycle, not just the final answer.

In [4]:
def ask_with_skills(agent_key, session_key, messages, show_events=True, content_preview_chars=160):
    """Send messages to an agent and print every skill-related event in the response.

    `messages` is the list passed straight to the API, e.g.
        [{"type": "text",  "content": "A customer just sent..."}]
    or  [{"type": "skill", "skill_name": "customer_escalation"}]
    or a mix of both.

    Returns the agent's final text reply (or None if there isn't one —
    e.g. when only a client-triggered skill_load was requested).
    """
    payload = {"messages": messages, "stream_response": False}
    url = f"{BASE_URL}/agents/{agent_key}/sessions/{session_key}/events"
    response = requests.post(url, headers=headers, json=payload)
    if response.status_code != 201:
        print(f"Error: {response.status_code} - {response.text}")
        return None

    events = response.json().get("events", [])
    if show_events:
        print("\n------ Agent Events ------")
        for event in events:
            etype = event.get("type", "unknown")
            if etype == "input_message":
                # Surface input shape so client-triggered skills are visible.
                kinds = [m.get("type") for m in (event.get("messages") or [])]
                print(f"[input_message] kinds={kinds}")
            elif etype == "tool_input" and event.get("tool_name") == "invoke_skill":
                # The model chose a skill — its argument lives in tool_input.
                print(f"[tool_input invoke_skill] args={event.get('tool_input', {})}")
            elif etype == "skill_load":
                name = event.get("skill_name", "?")
                content = event.get("content", "") or ""
                snippet = content[:content_preview_chars].replace("\n", " ")
                if len(content) > content_preview_chars:
                    snippet += "..."
                print(f"[skill_load] {name} ({len(content)} chars): {snippet}")
            # agent_output is printed by the caller after the helper returns.
        print("-" * 26)

    return next(
        (e.get("content") for e in events if e.get("type") == "agent_output"),
        None,
    )

In [5]:
# Create a session, then forward a real-looking customer message.
session_name = f"Skills Demo {datetime.now().strftime('%Y%m%d-%H%M%S')}"
response = requests.post(
    f"{BASE_URL}/agents/{skill_agent_key}/sessions",
    headers=headers,
    json={"name": session_name, "metadata": {"purpose": "skills_demo_autonomous"}},
)
response.raise_for_status()
session_key = response.json()["key"]
print(f"Session Created: {session_key}")

# Inbound message that should trigger `customer_escalation`: an outage with
# concrete impact language ("can't log in", "right now", "ETA").
incoming_outage_message = """From: ops@acme-financial.example
Subject: URGENT — none of our analysts can log in

Hi — it's about 9:14am ET and our entire analyst team (about 40 people)
is getting "Service unavailable" when they try to load dashboards. We're
in the middle of pre-market prep so this is blocking real work. We tried
two regions and it's the same. Can you tell us if you're aware of an
outage and give us an ETA? — Lina"""

print(f"\nInbound message:\n{incoming_outage_message}")
print("=" * 80)

answer = ask_with_skills(
    skill_agent_key,
    session_key,
    [{"type": "text", "content": f"A customer just sent us this. Help me handle it.\n\n{incoming_outage_message}"}],
)
print(f"\nCopilot reply:\n{answer}")

Session Created: ase_skills_demo_20260506-100816_5b26

Inbound message:
From: ops@acme-financial.example
Subject: URGENT — none of our analysts can log in

Hi — it's about 9:14am ET and our entire analyst team (about 40 people)
is getting "Service unavailable" when they try to load dashboards. We're
in the middle of pre-market prep so this is blocking real work. We tried
two regions and it's the same. Can you tell us if you're aware of an
outage and give us an ETA? — Lina

------ Agent Events ------
[input_message] kinds=['text']
[tool_input invoke_skill] args={'skill_name': 'customer_escalation'}
[skill_load] customer_escalation (2282 chars): When the inbound message looks like an escalation, follow this runbook exactly. Produce a single Markdown response with these sections, in order:  ## 1. Severit...
--------------------------

Copilot reply:
## 1. Severity classification
**SEV-1** — production is down for many customers. Anchoring phrases: "our entire analyst team (about 40 people

What just happened, in event order:

1. The model read the system prompt (with the *short* skill description) and decided this message needed `customer_escalation` — outage language, named impact, time pressure all match.
2. It emitted a `tool_input` for `invoke_skill` choosing that skill by name.
3. Vectara loaded the skill's `content` into the conversation as a user message and emitted the `skill_load` event you see in the trace.
4. The model then produced the structured response (severity classification, missing facts, routing, draft reply, internal note) using the loaded runbook.

If a customer had sent a routine "how do I export my dashboard?" message, that long runbook would never have entered context — every other turn stays cheap.

## Step 3: Client-Triggered Invocation

Sometimes you don't want the model to decide — you want to *put the skill in context up front*. A few realistic triggers in support-tool land:

- A monitoring webhook fired a SEV-1 alert and your bot should always start in escalation mode for that ticket.
- A routing rule already classified this email as "complaint"; you don't want to pay another LLM round-trip to re-decide.
- Your support UI has a "Treat as incident" button the human pressed before forwarding.

For those cases the API accepts an input message of type `"skill"`. You include it in the `messages` array of an event POST and Vectara loads the content immediately, no LLM round-trip needed for the decision.

In [6]:
# Fresh session so the prior skill_load doesn't already cover us.
response = requests.post(
    f"{BASE_URL}/agents/{skill_agent_key}/sessions",
    headers=headers,
    json={"name": f"Skills Client-Trigger {datetime.now().strftime('%Y%m%d-%H%M%S')}",
          "metadata": {"purpose": "skills_demo_client_trigger"}},
)
response.raise_for_status()
session_key = response.json()["key"]
print(f"Session Created: {session_key}")

# Step A: load the skill directly via a "skill" input. Pair it with a tiny
# placeholder text so the agent has something to acknowledge this turn.
print("\nStep A: client preloads the escalation skill (e.g. monitoring webhook fired SEV-1).")
print("=" * 80)
ask_with_skills(
    skill_agent_key, session_key,
    [
        {"type": "skill", "skill_name": "customer_escalation"},
        {"type": "text",  "content": "Incident ticket incoming. Acknowledge in one sentence and stand by."},
    ],
)

# Step B: now forward the actual customer message. The skill content is
# already in session history, so the agent uses it without re-invoking.
print("\nStep B: forward the message; runbook is already in context.")
print("=" * 80)
answer = ask_with_skills(
    skill_agent_key, session_key,
    [{"type": "text", "content": f"Here's the customer message:\n\n{incoming_outage_message}"}],
)
print(f"\nCopilot reply:\n{answer}")

Session Created: ase_skills_client-trigger_20260506-100819_9673

Step A: client preloads the escalation skill (e.g. monitoring webhook fired SEV-1).

------ Agent Events ------
[input_message] kinds=['skill', 'text']
[tool_input invoke_skill] args={'skill_name': 'customer_escalation'}
[skill_load] customer_escalation (2282 chars): When the inbound message looks like an escalation, follow this runbook exactly. Produce a single Markdown response with these sections, in order:  ## 1. Severit...
--------------------------

Step B: forward the message; runbook is already in context.

------ Agent Events ------
[input_message] kinds=['text']
--------------------------

Copilot reply:
## 1. Severity classification
**SEV-1** — Production is down for many customers. Phrases anchoring this decision: "none of our analysts can log in" and "blocking real work".

## 2. Required information to collect
- Account / tenant / org id
- Reproduction steps or confirmation that it affects all users
- Any err

Notice the trace difference vs. Step 2:

- **Step 2 (autonomous)**: a `tool_input` for `invoke_skill` *followed by* `skill_load` — the model decided.
- **This Step / part A (client-triggered)**: a `skill_load` but no `invoke_skill` `tool_input` — Vectara loaded the content because the request asked it to.
- **This Step / part B**: skill is already in session history; neither event appears, the model just uses the previously-loaded runbook.

Use client-triggered invocation when *you* know which skill to load (a routing rule, a webhook, an explicit UI action) and want to skip the decision round-trip; let the agent decide autonomously when the choice depends on what the customer actually said.

## Step 4: Multiple Skills, Model Picks the Right One

Most support copilots need more than one playbook. We'll add a second skill — `feature_request_intake` — that loads when the inbound message is asking for a new capability rather than reporting an incident. The agent now sees *two* short descriptions in its system prompt and picks one (or neither) per turn.

In [7]:
feature_request_intake_skill = {
    "description": (
        "Use when the inbound message is asking for a new feature, an integration, "
        "or a roadmap change — not an outage or bug. Loads a structured intake "
        "checklist for handing the request to product."
    ),
    "content": """When the inbound message is a feature request, capture it cleanly so product can triage it without coming back to the customer with five clarifying questions. Produce a single Markdown response with these sections:

## 1. One-line summary
Restate the request in a single sentence using the customer's own framing where possible. No solutionizing yet.

## 2. Required intake fields
Fill these out from the message. If a field is not stated, write `<unknown>` — do NOT invent. Product will follow up.
- **Persona**: who is asking (role / team)
- **Problem**: the underlying job-to-be-done in their words
- **Current workaround**: what they do today, even if it's manual
- **Frequency / volume**: how often this comes up, for how many users
- **Desired outcome**: what "done" looks like for them
- **Time sensitivity**: hard deadline (renewal, audit) vs. nice-to-have

## 3. Categorization
Tag the request with at most three of these labels (omit any that don't apply):
`integration`, `enterprise-controls`, `reporting`, `developer-experience`, `mobile`, `accessibility`, `cost-control`, `multilingual`, `other`.

## 4. Suggested filing
- Where this should go (`product backlog` / `partner-integrations queue` / `existing epic <name>`).
- Whether it warrants a discovery call or can be filed asynchronously.

## 5. Customer-facing reply (draft)
Acknowledge the request, restate what you heard, set expectation about the response timeline. Do NOT commit to building anything. Three sentences max."""
}

multi_skill_config = {
    **skill_agent_config,
    "description": "Support copilot with customer_escalation and feature_request_intake skills.",
    "skills": {
        "customer_escalation":    customer_escalation_skill,
        "feature_request_intake": feature_request_intake_skill,
    },
}

skill_agent_key = delete_and_create_agent(multi_skill_config, SKILL_AGENT_NAME)

# Two messages, very different shapes. Use a fresh session per message so each
# pick is independent of the other.
incoming_messages = [
    """From: pm@northwind.example
Subject: SSO with Okta?

Hi! We're loving the product, but our security team has asked us to roll
out SSO before we expand seats. Specifically, can we wire up Okta SAML?
We're not in a huge rush — would just like to know if it's on the roadmap
or if we should wait. Happy to be a design partner. — Sam""",
    """From: head-of-data@globex.example
Subject: dashboards just disappeared

Hey — most of our team's saved dashboards are showing as empty since this
morning. The titles are still there but the underlying data is gone. We
have a board meeting at 4pm and a few of those dashboards are what they
want to see. Please look ASAP. — Priya""",
]
for m in incoming_messages:
    response = requests.post(
        f"{BASE_URL}/agents/{skill_agent_key}/sessions",
        headers=headers,
        json={"name": f"Skills Multi {datetime.now().strftime('%Y%m%d-%H%M%S-%f')}",
              "metadata": {"purpose": "skills_demo_multi"}},
    )
    response.raise_for_status()
    session_key = response.json()["key"]

    print(f"\nInbound message:\n{m}")
    print("=" * 80)
    answer = ask_with_skills(
        skill_agent_key, session_key,
        [{"type": "text", "content": f"A customer just sent us this. Help me handle it.\n\n{m}"}],
    )
    print(f"\nCopilot reply:\n{answer}")

Deleted existing agent 'Support Copilot' (agt_support_copilot_efae)
Created agent 'Support Copilot' (key: agt_support_copilot_707e)

Inbound message:
From: pm@northwind.example
Subject: SSO with Okta?

Hi! We're loving the product, but our security team has asked us to roll
out SSO before we expand seats. Specifically, can we wire up Okta SAML?
We're not in a huge rush — would just like to know if it's on the roadmap
or if we should wait. Happy to be a design partner. — Sam

------ Agent Events ------
[input_message] kinds=['text']
[tool_input invoke_skill] args={'skill_name': 'feature_request_intake'}
[skill_load] feature_request_intake (1492 chars): When the inbound message is a feature request, capture it cleanly so product can triage it without coming back to the customer with five clarifying questions. P...
--------------------------

Copilot reply:
## 1. One-line summary
Request to integrate Okta SAML for Single Sign-On (SSO) as part of expanding their security setup.

## 2. Requ

The trace should show the model picking *different* skills for the two messages: `feature_request_intake` for the SSO/Okta ask, `customer_escalation` for the missing-data outage. With both descriptions visible in the system prompt, the model has the information it needs to route per turn — you don't have to write classifier code.

## Step 5: Restricting Skills per Step with `allowed_skills`

For multi-step agents (notebook 9), each step independently controls which skills are exposed via `allowed_skills`:

- **omitted (`null`)** — default; all skills the agent declares are available
- **empty array (`[]`)** — no skills available in this step; the `invoke_skill` tool isn't exposed at all
- **list of names** — only those skills are loadable here

This pairs cleanly with the multi-step pattern: a *router* step might forbid skills entirely (force routing-only behavior), then transition into a *specialist* step that exposes exactly the right one. For example: a router step decides whether the message is an incident vs. a feature ask, then transitions to one of two follow-up steps that each expose only the matching skill.

Below is the *shape* of step configs with `allowed_skills`. We don't build a full multi-step agent here — see notebook 9 for that orchestration pattern.

In [8]:
# Illustrative only — no API call. Three forms of `allowed_skills`:
allowed_skills_examples = {
    "all_skills_available_default": {
        "instructions": [{"type": "inline", "template": "Generalist support step."}],
        "output_parser": {"type": "default"},
        # `allowed_skills` omitted → all agent skills available
    },
    "incident_specialist_step": {
        "instructions": [{"type": "inline", "template": "Handle incident-class messages only."}],
        "output_parser": {"type": "default"},
        "allowed_skills": ["customer_escalation"],
    },
    "router_step_no_skills": {
        "instructions": [{"type": "inline", "template": "Route the message; do not draft a reply."}],
        "output_parser": {"type": "default"},
        "allowed_skills": [],   # invoke_skill tool not exposed in this step
    },
}
print(json.dumps(allowed_skills_examples, indent=2))

{
  "all_skills_available_default": {
    "instructions": [
      {
        "type": "inline",
        "template": "Generalist support step."
      }
    ],
    "output_parser": {
      "type": "default"
    }
  },
  "incident_specialist_step": {
    "instructions": [
      {
        "type": "inline",
        "template": "Handle incident-class messages only."
      }
    ],
    "output_parser": {
      "type": "default"
    },
    "allowed_skills": [
      "customer_escalation"
    ]
  },
  "router_step_no_skills": {
    "instructions": [
      {
        "type": "inline",
        "template": "Route the message; do not draft a reply."
      }
    ],
    "output_parser": {
      "type": "default"
    },
    "allowed_skills": []
  }
}


## Step 6: When to Reach for a Skill (Decision Aid)

- **Skill** — your agent should sometimes adopt a different mindset/checklist/voice that's too long to keep in the system prompt every turn. The set of mindsets is small, fixed, and you can describe each one in a sentence or two. *(Today's notebook.)*
- **Tool** — the agent needs to *do* something (search a corpus with `corpora_search`, call an API with `web_get`, run code with `lambda`). Skills load *instructions*; they don't take actions.
- **Sub-agent** — the work needs its own model, its own tool set, or its own multi-step flow. Heavier than a skill — see notebook 5.
- **Just stuff the prompt** — the guidance applies to *every* turn and is short enough that lazy loading is more confusing than it is cheap.

A common pairing in real Vectara deployments: a support copilot uses `corpora_search` (over your help-center / runbook corpus) to *retrieve* facts, AND skills to switch into the right *workflow* for each message. The corpus answers "what's true," the skill answers "what should I do about it."

### Anti-patterns

- **Skills as content stores.** A skill is *instructions*. If you find yourself stuffing reference data ("here are 200 product SKUs", "the full pricing matrix", "every error code") into `content`, you want a corpus and `corpora_search`, not a skill.
- **Too many skills.** If the model has 15 skill descriptions to read every turn, the system prompt is bloated again — and selection accuracy drops. Combine related skills, or move them under sub-agents.
- **Vague descriptions.** "Helps with reviews" under-fires because the model can't tell when to reach for it. Make descriptions directive: "**Use when** the inbound message X. Loads Y."
- **Forgetting the cap.** `description` is capped at 500 chars and `content` at 50,000 chars. Long runbooks should be sliced into a few smaller skills, not one giant one.

## Cleanup (Optional)

Delete the agent created in this notebook so you don't accumulate test resources.

In [9]:
if skill_agent_key:
    response = requests.delete(f"{BASE_URL}/agents/{skill_agent_key}", headers=headers)
    if response.status_code == 204:
        print(f"Deleted Support Copilot agent: {skill_agent_key}")
    else:
        print(f"Error deleting {skill_agent_key}: {response.status_code} - {response.text}")

Deleted Support Copilot agent: agt_support_copilot_707e
